# Biohub Lab HOCT Morphology Diagnostic

**Measured result, version 1: negative.** On four in-sample training videos,
this implementation scored 0.9195083 versus 0.9666951 for its fixed-node control.
Evaluated false divisions rose from 1 to 12. Preserved for reproducibility;
**this direct-transfer configuration is not recommended for submission**.

New architecture: image-derived 3D watershed masks → intensity/inertia features →
pretrained [HOCT](https://github.com/royerlab/hoct) general_v1 → exact capacitated matching.
Retains the fixed control detections; replaces its association and division logic.
The masks are approximate intensity basins, not FOCUS-3D predictions.

HOCT commit `2ccc5040823bc944ab67790abd1f56eea7cd4f05`, MIT license included.
Our matcher uses consecutive edges and the fixed-node single-pass objective;
this is **not** the upstream two-pass tracklet solver. FP32 supports Kaggle T4.

Four complete training videos, all known to be in the secondary detector training set. This is an in-sample diagnostic, not an unbiased validation score. Reuses cached detections and never exports a root submission.

Detector provenance: public Pilkwang checkpoints and Igor Zharov/flexonafft Harmonic Fusion.
All final CSVs are checked and diagnostic scores use the pinned official metric.


In [ ]:
import base64, hashlib, io, os, subprocess, sys, zipfile
from pathlib import Path
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'a28459492e2cdd3407bd7bbe4b81d2bc0b63c4bb132743b0ece4346efa0f02ef'
package = Path('/kaggle/working/hoct_package')
package.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for member in archive.namelist():
        if not (package/member).resolve().is_relative_to(package.resolve()):
            raise ValueError('Unsafe package path')
    archive.extractall(package)
env = dict(os.environ, PYTHONPATH=str(package/'src'))
mode = 'diagnostic'
if mode == 'diagnostic':
    source = (package/'baseline/harmonic_inference.py').read_text()
    marker = '\nmaterialize_inference_repo(ARTIFACTS)'
    assert source.count(marker) == 1
    bootstrap = package/'bootstrap.py'
    bootstrap.write_text(source.split(marker)[0])
    subprocess.run([sys.executable,'-u',str(bootstrap)],env=env,check=True)
else:
    subprocess.run([sys.executable,'-u',str(package/'scripts/notebook_runner.py'),
        'control',str(package)],env=env,check=True)
    Path('/kaggle/working/submission.csv').rename('/kaggle/working/control_for_comparison.csv')
subprocess.run([sys.executable,'-u',str(package/'scripts/hoct_runner.py'),
    mode,str(package)],env=env,check=True)
